In [ ]:
from IPython.display import display, Javascript

display(Javascript('''
    var output_areas = this.notebook.get_cells().map(function(cell) {
        return cell.output_area;
    });
    output_areas.forEach(function(output_area) {
        if (output_area) {
            output_area.expand();
        }
    });
'''))

#### 读取对比 _cntrl_ r1, r2, r3 文件内容区别

In [ ]:
from collections import deque
data_paths = \
    ['/data/fangping/bulleye/Bullseye UPenn data/BE_cntrl_r3_counts.DNA.txt David Orlando',
     '/data/fangping/bulleye/Bullseye UPenn data/BE_cntrl_r2_counts.DNA.txt David Orlando',
     '/data/fangping/bulleye/Bullseye UPenn data/BE_cntrl_r1_counts.DNA.txt David Orlando']

# 定义要读取的行数
front_lines_to_read = 20
end_lines_to_read = 8

# 打开文件并读取指定行数
for data_path in data_paths:
    print('\n')
    with open(data_path, 'r', encoding='utf-8') as file:
        for i in range(front_lines_to_read):
            line = file.readline()  # 逐行读取
            if not line:  # 文件读取结束时退出循环
                break
            print(f"{line.strip()}")

    print(f'⬆ first {front_lines_to_read} lines\n.............\n⬇ last {end_lines_to_read} lines')

    with open(data_path, "r", encoding="utf-8") as file:
        last_lines = deque(file, maxlen=end_lines_to_read)  # 固定大小队列
        for line in last_lines:
            print(line.strip())

#### 读取对比同一个 replicate 不同文件内容区别

In [ ]:
data_paths = \
    ['/data/fangping/bulleye/Bullseye UPenn data/BE_cntrl_r1_regionInfo.txt David Orlando',
     '/data/fangping/bulleye/Bullseye UPenn data/BE_cntrl_r1_counts.peptide.txt David Orlando',
     '/data/fangping/bulleye/Bullseye UPenn data/BE_cntrl_r1_counts.DNA.txt David Orlando']

# 定义要读取的行数
lines_to_read = 20

# 打开文件并读取指定行数
for data_path in data_paths:
    print('\n')
    with open(data_path, 'r', encoding='utf-8') as file:
        for i in range(lines_to_read):
            line = file.readline()  # 逐行读取
            if not line:  # 文件读取结束时退出循环
                break
            print(f"{line.strip()}")

#### 处理同一阶段不同 replicate 成字典

In [ ]:
import pickle

data_path_formats = \
    ['/data/fangping/bulleye/Bullseye UPenn data/BE_{}_r3_counts.DNA.txt David Orlando',
     '/data/fangping/bulleye/Bullseye UPenn data/BE_{}_r2_counts.DNA.txt David Orlando',
     '/data/fangping/bulleye/Bullseye UPenn data/BE_{}_r1_counts.DNA.txt David Orlando']

save_path_formats = \
    ['/data/fangping/bulleye/Bullseye UPenn dict data/BE_{}_r3_counts.DNA.pkl',
     '/data/fangping/bulleye/Bullseye UPenn dict data/BE_{}_r2_counts.DNA.pkl',
     '/data/fangping/bulleye/Bullseye UPenn dict data/BE_{}_r1_counts.DNA.pkl']

categories = ['cntrl', 'log', 'stat']

for category in categories:
    for i, data_path in enumerate(data_path_formats):
        complete_data_path = data_path.format(category)
        content_dict = {}
        print(f'processing {complete_data_path}')
        with open(complete_data_path, 'r', encoding='utf-8') as file:
            for line in iter(file.readline, ''):  # 迭代直到文件结束
                if line.startswith('#') or len(line.strip()) == 0:
                    continue
                contents = line.strip().split()
                if any(item in contents[1] for item in ['*', '_', 'B', 'J', 'O', 'U', 'X', 'Z', 'b', 'j', 'o', 'u', 'x', 'z', 'Peptide']):
                    continue
                else:
                    pep_seq = contents[1].strip()
                    if pep_seq == 'CIVYLPD':
                        print(f'{pep_seq}: count: {int(contents[2])}')
                    if content_dict.get(pep_seq, None) is None:
                        content_dict[pep_seq] = {}
                    if content_dict[pep_seq].get('Count', None) is not None:
                        content_dict[pep_seq]['Count'].append(int(contents[2]))
                    else:
                        content_dict[pep_seq]['Count'] = [int(contents[2])]
                    if content_dict[pep_seq].get('DNA', None) is not None:
                        content_dict[pep_seq]['DNA'].append(contents[0])
                    else:
                        content_dict[pep_seq]['DNA'] = [contents[0]]

        # 保存字典
        complete_save_path = save_path_formats[i].format(category)
        with open(complete_save_path, 'wb') as file:
            pickle.dump(content_dict, file)
            print(f'completed {complete_save_path}')

In [ ]:
import pickle

data_path_formats = \
    ['/data/fangping/bulleye/Bullseye_UPenn_data/BE_{}_r3_counts.DNA.txt David Orlando',
     '/data/fangping/bulleye/Bullseye_UPenn_data/BE_{}_r2_counts.DNA.txt David Orlando',
     '/data/fangping/bulleye/Bullseye_UPenn_data/BE_{}_r1_counts.DNA.txt David Orlando']

save_path_formats = \
    ['/data/fangping/bulleye/Bullseye UPenn dict data/BE_{}_r3_counts.DNA.pkl',
     '/data/fangping/bulleye/Bullseye UPenn dict data/BE_{}_r2_counts.DNA.pkl',
     '/data/fangping/bulleye/Bullseye UPenn dict data/BE_{}_r1_counts.DNA.pkl']

categories = ['cntrl', 'log', 'stat']

for category in categories:
    for i, data_path in enumerate(data_path_formats):
        complete_data_path = data_path.format(category)
        content_dict = {}
        print(f'processing {complete_data_path.split('/')[-1]}')
        with open(complete_data_path, 'r', encoding='utf-8') as file:
            for line in iter(file.readline, ''):  # 迭代直到文件结束
                if line.startswith('#') or len(line.strip()) == 0:
                    continue
                contents = line.strip().split()
                if any(item in contents[1] for item in ['*', '_', 'B', 'J', 'O', 'U', 'X', 'Z', 'b', 'j', 'o', 'u', 'x', 'z', 'Peptide']):
                    continue
                else:
                    if contents[-1] == 0:
                        print(contents)

#### 计算三个 replicate 合并 Count 字典

In [ ]:
import pickle
from tqdm import tqdm
import os
import json

dict_file_path_format = '/data/fangping/bulleye/Bullseye UPenn dict data/BE_{}_r{}_counts.DNA.pkl'
output_file_path_format = '/data/fangping/bulleye/Bullseye UPenn dict data/BE_{}_merged_counts.DNA.json'

categories = ['cntrl', 'log', 'stat']

replicate_num = 3

for category in categories:
    dicts = []
    merged_dict = {}
    if os.path.exists(output_file_path_format.format(category)):
        print(f'{output_file_path_format.format(category).split("/")[-1]} already exists, skipping...')
        continue
    for i in range(replicate_num):
        complete_data_path = dict_file_path_format.format(category, i+1)
        with open(complete_data_path, 'rb') as file:
            print(f'loading {complete_data_path.split("/")[-1]}...')
            dicts.append(pickle.load(file))
            print(f'loaded {complete_data_path.split("/")[-1]}')

    # 找三个 dict.keys() 的交集
    common_keys = set(dicts[0].keys()).intersection(*(d.keys() for d in dicts[1:]))
    for key in tqdm(common_keys, desc=f"Merging {category} Counts"):
        merged_dict[key] = {
            'Count': [],
            'DNA': []
        }
        # 合并 Count 和 DNA 列表
        for d in dicts:
            merged_dict[key]['Count'].extend(d[key]['Count'])
            merged_dict[key]['DNA'].extend(d[key]['DNA'])

    # 保存结果到一个新的 pkl 文件
    output_file = output_file_path_format.format(category)
    with open(output_file, 'w') as f:
        print(f'saving {output_file.split("/")[-1]}...')
        json.dump(merged_dict, f, indent=4)

    print(f"Merged dictionary saved to {output_file}")

#### 看一下供 apex 使用的文件对不对

In [ ]:
file_path = '/data/fangping/bulleye/Bullseye UPenn dict data/BE_cntrl_peptides_seqs.txt'

# 定义要读取的行数
lines_to_read = 10

# 打开文件并读取前几行
with open(file_path, 'r', encoding='utf-8') as file:
    for i in range(lines_to_read):
        line = file.readline()
        if not line:  # 文件读取结束时退出循环
            break
        print(line.strip())

In [ ]:
file_path = '/data/fangping/bulleye/Bullseye UPenn dict data/BE_cntrl_peptides_seqs.txt'

with open(file_path, 'r', encoding='utf-8') as file:
    line_count = sum(1 for _ in file)

print(f"The file {file_path.split("/")[-1]} has {line_count} lines.")

In [ ]:
print(f'{32167374/3000}')

#### 读取展示 APEX 预测结果前几行

In [ ]:
import csv

file_path = "/data/fangping/bulleye/APEX_results/Predicted_MICs_cntrl.csv"

num_rows = 10
with open(file_path, mode="r", encoding="utf-8") as file:
    reader = csv.reader(file)
    for i, row in enumerate(reader):
        print(row)
        if i + 1 == num_rows:
            break

#### APEX 预测求平均值示例

In [ ]:
import csv
from tqdm import tqdm
import pickle

# 文件路径
file_path = "/data/fangping/bulleye/APEX_results/Predicted_MICs_log.csv"

# 初始化一个空字典存储结果
result = []

# 读取文件并计算平均值
with open(file_path, mode="r", encoding="utf-8") as file:
    reader = csv.reader(file)

    # 遍历每一行
    for i, row in tqdm(enumerate(reader)):
        # if i==10:
        #     break
        # 跳过空行或不符合格式的行
        if not row or len(row) < 2:
            continue

        # 第一列作为键，后面的数值作为值
        key = row[0]
        try:
            # 将后续列的数值转换为浮点数，并计算平均值
            values = [float(x) for x in row[1:] if x]
            average = sum(values) / len(values)
            result.append(average)
            # result_dict[key] = average
        except ValueError:
            # 跳过无法转换为数值的行
            print(f"警告：行数据无法处理，已跳过：{row}")

# 输出结果
print(len(result))
print('saving result')
with open("/data/fangping/bulleye/APEX_results/log_average_mic.pkl", "wb") as f:
    pickle.dump(result, f)
print('saved')

#### 存成字典形式的

In [ ]:
import csv
from tqdm import tqdm
import pickle

# 文件路径
file_path = "/data/fangping/bulleye/APEX_results/Predicted_MICs_stat.csv"

# 初始化一个空字典存储结果
result_dict = {}

# 读取文件并计算平均值
with open(file_path, mode="r", encoding="utf-8") as file:
    reader = csv.reader(file)

    # 遍历每一行
    for i, row in tqdm(enumerate(reader)):
        # if i==10:
        #     break
        # 跳过空行或不符合格式的行
        if not row or len(row) < 2:
            continue

        # 第一列作为键，后面的数值作为值
        key = row[0]
        try:
            # 将后续列的数值转换为浮点数，并计算平均值
            values = [float(x) for x in row[1:] if x]
            average = sum(values) / len(values)
            # result.append(average)
            result_dict[key] = average
        except ValueError:
            # 跳过无法转换为数值的行
            print(f"警告：行数据无法处理，已跳过：{row}")

# 输出结果
# print(len(result))
print('saving result')
with open("/data/fangping/bulleye/APEX_results/stat_average_mic_dict.pkl", "wb") as f:
    pickle.dump(result_dict, f)
print('saved')

#### r1, r2, r3 都重合的 peptide 序列.txt转换成 fasta 格式方便ESM

In [ ]:
input_file_format = '/data/fangping/bulleye/Bullseye UPenn dict data/BE_{}_peptides_seqs.txt'
output_file_format = '/data/fangping/bulleye/Bullseye UPenn dict data/BE_{}_peptides_seqs.fasta'

def txt_to_fasta(input_file, output_file):
    """
    将每行包含氨基酸序列的.txt文件转换为FASTA格式的文件。

    Args:
        input_file (str): 输入的.txt文件路径。
        output_file (str): 输出的FASTA格式文件路径。
    """
    try:
        with open(input_file, 'r') as infile, open(output_file, 'w') as outfile:
            for i, line in enumerate(infile):
                sequence = line.strip()  # 去掉行首和行尾的空格或换行符
                if sequence:  # 确保非空行
                    outfile.write(f">{sequence}\n")
                    outfile.write(sequence + "\n")
        print(f"转换成功！输出已保存到 {output_file}")
    except FileNotFoundError:
        print(f"文件未找到：{input_file}")
    except Exception as e:
        print(f"发生错误：{e}")

for kind in ['cntrl', 'log', 'stat']:
    # 示例使用
    input_txt = input_file_format.format(kind)  # 输入文件路径
    output_fasta = output_file_format.format(kind)  # 输出文件路径
    txt_to_fasta(input_txt, output_fasta)

#### 重新处理所有的 在 cntrl, log, stat 的 r1,r2,r3 中都存在的 DNA
*注：使用外部文件：DNA_reads_all.py

In [ ]:
import numpy as np
import pickle
from tqdm import tqdm
import json
import pandas as pd

data_path_format = '/data/fangping/bulleye/Bullseye_UPenn_data/BE_{}_r{}_counts.DNA.txt David Orlando'

save_path = '/data/fangping/bulleye/Bullseye_UPenn_dict_data/BE_cntrl_log_stat_merged_counts.DNA.csv'

categories = ['cntrl', 'log', 'stat']

all_DNA_reads_dict = {}

for complete_data_path in tqdm([data_path_format.format(category, repeat_num) for category in categories for repeat_num in range(1, 4)], desc=' processing files...'):
    print(f'processing {complete_data_path}')
    with open(complete_data_path, 'r', encoding='utf-8') as file:
        for line in iter(file.readline, ''):  # 迭代直到文件结束
            if line.startswith('#') or len(line.strip()) == 0:
                continue
            contents = line.strip().split()
            if any(item in contents[0] for item in ['_', 'DNA']):
                continue
            else:
                DNA_seq = contents[0].strip()
                if DNA_seq == 'TGCATTGTGTACCTGCCTGAT':
                    print(f'{DNA_seq}: count: {int(contents[2])}')
                if all_DNA_reads_dict.get(DNA_seq, None) is None:
                    all_DNA_reads_dict[DNA_seq] = []
                else:
                    all_DNA_reads_dict[DNA_seq].append(int(contents[2]))

# 用于存储成 csv 文件
all_DNA_reads_list = []
for DNA_seq, reads_list in tqdm(all_DNA_reads_dict.items()):
    if len(reads_list) == 9:
        meaned_reads = [] # shaoe: (3)
        reads_list = np.array(reads_list)
        step_size = 3
        for step in range(3):
            meaned_reads.append(reads_list[step*3:step*3+step_size].mean())
        all_DNA_reads_list.append([DNA_seq, meaned_reads.tolist()])
    elif len(reads_list) > 9:
        print(f'unexpected length {len(reads_list)}')
        exit(1)

df = pd.DataFrame(all_DNA_reads_list, columns=['DNA_seq', 'meaned_reads'])
df.to_csv(save_path, index=False)



#### sum up count r1 peptide reads

In [ ]:
with open('/data/fangping/bulleye/Bullseye_UPenn_data/BE_cntrl_r1_counts.DNA.txt David Orlando', 'r', encoding='utf-8') as file:
    total_reads = 0
    for line in iter(file.readline, ''):  # 迭代直到文件结束
        if line.startswith('#') or len(line.strip()) == 0:
            continue
        contents = line.strip().split()
        # 去掉那些 peptide 有问题的
        # if any(item in contents[1] for item in ['*', '_', 'B', 'J', 'O', 'U', 'X', 'Z', 'b', 'j', 'o', 'u', 'x', 'z', 'Peptide']):
        if any(item in contents[1] for item in ['_', 'Peptide']):
            continue
        else:
            total_reads += int(contents[2])

print(f'total reads: {total_reads}')